# Bronze Layer Exploration
Raw events written directly from Kafka. No cleaning or deduplication yet.
Schema is set at ingest time so we can inspect exactly what the simulator produces.

In [ ]:
import sys
sys.path.insert(0, '..')  # run from notebooks/ directory

from storage.spark_session import build_delta_spark
from storage.delta_config import BRONZE_RIDES_TABLE, BRONZE_DRIVERS_TABLE, BRONZE_PAYMENTS_TABLE

spark = build_delta_spark('BronzeExploration')
print('Spark version:', spark.version)

## 1. Rides Bronze

In [ ]:
rides = spark.read.format('delta').load(BRONZE_RIDES_TABLE)
print(f'Row count: {rides.count():,}')
rides.printSchema()

In [ ]:
rides.show(5, truncate=False)

In [ ]:
# Check for duplicates — Bronze should be append-only so duplicates are expected
# when the simulator restarts or when late events arrive.
total = rides.count()
distinct = rides.dropDuplicates(['ride_id']).count()
print(f'Total rows: {total:,} | Distinct ride_ids: {distinct:,} | Duplicates: {total - distinct:,}')

In [ ]:
# Status distribution
rides.groupBy('status').count().orderBy('count', ascending=False).show()

In [ ]:
# Surge distribution by zone
from pyspark.sql import functions as F

rides.groupBy('city_zone').agg(
    F.count('ride_id').alias('ride_count'),
    F.round(F.avg('surge_multiplier'), 2).alias('avg_surge'),
    F.min('surge_multiplier').alias('min_surge'),
    F.max('surge_multiplier').alias('max_surge'),
).orderBy('ride_count', ascending=False).show()

## 2. Drivers Bronze

In [ ]:
drivers = spark.read.format('delta').load(BRONZE_DRIVERS_TABLE)
print(f'Row count: {drivers.count():,}')
drivers.groupBy('status').count().show()

## 3. Delta Time Travel — what did the table look like 1 hour ago?

In [ ]:
# Delta Lake time travel via timestamp
from datetime import datetime, timedelta, timezone

one_hour_ago = (datetime.now(timezone.utc) - timedelta(hours=1)).isoformat()

try:
    past_rides = (
        spark.read.format('delta')
        .option('timestampAsOf', one_hour_ago)
        .load(BRONZE_RIDES_TABLE)
    )
    print(f'Row count 1h ago: {past_rides.count():,}')
except Exception as exc:
    print(f'Time travel not available yet (table may be too new): {exc}')

In [ ]:
# Delta table history
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, BRONZE_RIDES_TABLE)
dt.history(10).select('version', 'timestamp', 'operation', 'operationMetrics').show(truncate=False)